# Challenge 1 - Tic Tac Toe



In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).



There are 9 grids in Tic Tac Toe that are coded as the following picture shows:



![Tic Tac Toe Grids](tttboard.jpg)



In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.



## Step 1: Data Engineering



This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.



1. Read `tic-tac-toe.csv` into a dataframe.

1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.

1. Convert the categorical values to numeric in all columns.

1. Separate the inputs and output.

1. Normalize the input data.

In [2]:
from google.colab import files
uploaded = files.upload()

Saving tic-tac-toe.csv to tic-tac-toe.csv


In [3]:
# Step 1: Data Engineering

import pandas as pd
import numpy as np

# 1. Read tic-tac-toe.csv into a dataframe
data = pd.read_csv('tic-tac-toe.csv')

# 2. Inspect the dataset
print(data.shape)
print(data.dtypes)
print(data.isnull().sum())
data.head()

(958, 10)
TL       object
TM       object
TR       object
ML       object
MM       object
MR       object
BL       object
BM       object
BR       object
class      bool
dtype: object
TL       0
TM       0
TR       0
ML       0
MM       0
MR       0
BL       0
BM       0
BR       0
class    0
dtype: int64


,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True


In [4]:
# Check class balance and unique values per column (sanity check on data reliability)
print(data['class'].value_counts())

for col in data.columns:
    print(col, data[col].unique())

class
True     626
False    332
Name: count, dtype: int64
TL ['x' 'o' 'b']
TM ['x' 'o' 'b']
TR ['x' 'o' 'b']
ML ['x' 'o' 'b']
MM ['o' 'b' 'x']
MR ['o' 'b' 'x']
BL ['x' 'o' 'b']
BM ['o' 'x' 'b']
BR ['o' 'x' 'b']
class [ True False]


In [5]:
# 3. Convert categorical values to numeric in all columns
# 'x', 'o', 'b' -> numeric codes; class True/False -> 1/0
data_encoded = data.copy()

for col in data_encoded.columns[:-1]:  # the 9 board columns
    data_encoded[col] = data_encoded[col].map({'x': 1, 'o': -1, 'b': 0})

data_encoded['class'] = data_encoded['class'].astype(int)  # True->1, False->0

data_encoded.head()

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,1,1,1,1,-1,-1,1,-1,-1,1
1,1,1,1,1,-1,-1,-1,1,-1,1
2,1,1,1,1,-1,-1,-1,-1,1,1
3,1,1,1,1,-1,-1,-1,0,0,1
4,1,1,1,1,-1,-1,0,-1,0,1


In [6]:
# 4. Separate the inputs and output
X = data_encoded.drop('class', axis=1)
y = data_encoded['class']

print(X.shape, y.shape)

(958, 9) (958,)


In [7]:
# 5. Normalize the input data
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

print(X_normalized[:3])

[[ 1.03516957  1.10682993  1.03516957  1.10682993 -1.24199419 -1.2235944
   1.03516957 -1.2235944  -1.23155602]
 [ 1.03516957  1.10682993  1.03516957  1.10682993 -1.24199419 -1.2235944
  -1.23155602  1.10682993 -1.23155602]
 [ 1.03516957  1.10682993  1.03516957  1.10682993 -1.24199419 -1.2235944
  -1.23155602 -1.2235944   1.03516957]]


## Step 2: Build Neural Network



To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.



1. Split the training and test data.

1. Create a `Sequential` model.

1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.

1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.

1. Fit the training data.

1. Evaluate your neural network model with the test data.

1. Save your model as `tic-tac-toe.model`.

In [8]:
# Step 2: Build Neural Network

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split

# 1. Split the training and test data
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(766, 9) (192, 9)


In [9]:
# 2-3. Create a Sequential model with several Dense layers
# ReLU for the hidden layers, Softmax for the output layer (2 classes: win / not win)
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(2, activation='softmax')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,786 (10.88 KB)

 Trainable params: 2,786 (10.88 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# 4. Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
# 5. Fit the training data
history = model.fit(
    X_train, y_train,
    epochs=50,
    validation_split=0.1,
    verbose=1
)

Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - accuracy: 0.6357 - loss: 0.6219 - val_accuracy: 0.7792 - val_loss: 0.5405
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7460 - loss: 0.5459 - val_accuracy: 0.7792 - val_loss: 0.4987
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7837 - loss: 0.5034 - val_accuracy: 0.7922 - val_loss: 0.4743
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7939 - loss: 0.4719 - val_accuracy: 0.7922 - val_loss: 0.4510
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8070 - loss: 0.4399 - val_accuracy: 0.7922 - val_loss: 0.4325
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8302 - loss: 0.4090 - val_accuracy: 0.8052 - val_loss: 0.4050
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8607 - loss: 0.3716 - val_accuracy: 0.8701 - val_loss: 0.3634
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8708 - loss: 0.3363 - val_accuracy: 0.8961 - val_loss

In [12]:
# 6. Evaluate the model with the test data
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

Test loss: 0.0336
Test accuracy: 0.9844


In [13]:
# 7. Save the model
model.save('tic-tac-toe.model.keras')
print("Model saved.")

Model saved.


## Step 3: Make Predictions



Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [14]:
# Step 3: Make Predictions

import random
from tensorflow.keras.models import load_model

# Load the saved model
loaded_model = load_model('tic-tac-toe.model.keras')

# Pick a few random rows from the test dataset
random_indices = random.sample(range(len(X_test)), 5)
sample_X = X_test[random_indices]
sample_y_true = y_test.iloc[random_indices].values

# Predict
predictions = loaded_model.predict(sample_X)
predicted_classes = predictions.argmax(axis=1)

for i in range(len(random_indices)):
    print(f"True label: {sample_y_true[i]} | Predicted: {predicted_classes[i]} | Correct: {sample_y_true[i] == predicted_classes[i]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step
True label: 0 | Predicted: 0 | Correct: True
True label: 0 | Predicted: 0 | Correct: True
True label: 0 | Predicted: 0 | Correct: True
True label: 0 | Predicted: 0 | Correct: True
True label: 1 | Predicted: 1 | Correct: True


## Step 4: Improve Your Model



Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.



But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.



* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.

* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.

    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).

    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.

* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [15]:
# Step 4: Improve Your Model

# Attempt 1: more layers + custom learning rate + more epochs
from tensorflow.keras.optimizers import Adam

model_v2 = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(2, activation='softmax')
])

custom_adam = Adam(learning_rate=0.001)

model_v2.compile(
    optimizer=custom_adam,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_v2 = model_v2.fit(
    X_train, y_train,
    epochs=100,
    validation_split=0.1,
    verbose=1
)

test_loss_v2, test_accuracy_v2 = model_v2.evaluate(X_test, y_test, verbose=0)
print(f"Improved model - Test loss: {test_loss_v2:.4f}")
print(f"Improved model - Test accuracy: {test_accuracy_v2:.4f}")

# Save the improved model
model_v2.save('tic-tac-toe.model.keras')

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 71ms/step - accuracy: 0.6705 - loss: 0.6097 - val_accuracy: 0.8312 - val_loss: 0.5121
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7736 - loss: 0.5039 - val_accuracy: 0.8052 - val_loss: 0.4400
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8099 - loss: 0.4243 - val_accuracy: 0.8442 - val_loss: 0.3659
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8636 - loss: 0.3383 - val_accuracy: 0.8571 - val_loss: 0.3097
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9536 - loss: 0.2379 - val_accuracy: 0.9610 - val_loss: 0.2148
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9884 - loss: 0.1443 - val_accuracy: 0.9870 - val_loss: 0.1530
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9927 - loss: 0.0806 - val_accuracy: 0.9870 - val_loss: 0.1117
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9985 - loss: 0.0438 - val_accuracy: 0.9870 - 

**Which approach(es) did you find helpful to improve your model performance?**

Which approach(es) did you find helpful to improve your model performance?

The original model (Step 2: 2 hidden layers, 50 epochs, default Adam) already met both
target thresholds on its own: test loss 0.0336 (< 0.1) and test accuracy 0.9844 (> 0.95).

The "improved" model (Step 4: 3 hidden layers, 100 epochs, explicit learning_rate=0.001)
only produced a marginal reduction in test loss (0.0336 → 0.0257), while test accuracy
stayed exactly the same (0.9844). This suggests that for this dataset — which has a
relatively small, well-structured feature space (9 board positions with clear geometric
win patterns) — the simpler model was already close to the practical performance ceiling.
Adding more layers, more epochs, and tuning the learning rate offered diminishing returns
rather than a meaningful improvement, which is itself a useful takeaway: more complexity
doesn't always translate into better results, especially once a model has already
captured the core patterns in the data.